<a href="https://colab.research.google.com/github/mshinno26/UnderstandingAI/blob/main/KNN_Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [106]:
pip install ucimlrepo # access to our datasets

In [107]:
import numpy as np # a Python library for calculations

In [108]:
class KNN:
    def __init__(self, k=5, task='classification'):
        self.k = k
        self.task = task

    def fit(self, X_train, y_train): #training function
        self.X_train = X_train
        self.y_train = y_train

    def set_k(self, k):
        self.k = k

    def set_task(self, task):
        self.task = task

    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

        # n = len(x1)
        # sum = 0
        # for i in range(n):
        #     diff = x1[i]-x2[i]
        #     sum += np.power(diff,2)
        # return np.sqrt(sum)

    def predict(self, X_test): #predict function
        predictions = [self.calculate_prediction(x) for x in X_test]
        return np.array(predictions)

    def calculate_prediction(self, x):
        distances = []
        for x_train in self.X_train:
            distance = self.euclidean_distance(x_train, x)
            distances.append(distance)
        #make a new list, with the indices of the k smallest elements of distances
        neighbors = np.argsort(distances)[:self.k]
        targets = []
        closest = []
        for i in neighbors:
            targets.append(self.y_train[i])
            closest.append(distances[i])

        # for i in range(self.k):
        #     weight = (self.k-i)/denominator
        #     print(targets[i])
        #     avg += weight*float(targets[i])
        # return int(round(avg))

        if self.task=='classification':
            #returns the mode
            unique, counts = np.unique(targets, return_counts=True)
            return unique[np.argmax(counts)]
        elif self.task=='regression':
            return int(round(np.mean(targets)))
        elif self.task=='ordered regression':
            #weights are based on order of closeness, not the actual distances
            denominator = np.sum(np.array(range(self.k, 0, -1)))
            return int(round(np.sum(np.array([(self.k-i)/denominator*targets[i] for i in range(self.k)]))))
        elif self.task=='weighted regression':
            #weights are based directly on the actual distance to the point
            denominator = np.sum(distances)
            return int(round(np.sum(np.array([(denominator-distances[i])/denominator*targets[i] for i in range(self.k)]))))
        else:
            raise ValueError("Unknown task passed to class KNN(); task must be 'classification', 'regression', or 'weighted regression.")

        #draft 1: doesn't work because duplicate distances
        # neighbors = {}
        # for i in range(len(self.X_train)):
        #     train = self.X_train[i]
        #     train_ring = self.y_train[i][0]
        #     d = self.euclidean_distance(train, x)
        #     if len(neighbors) < self.k:
        #         neighbors.update({d:train_ring})
        #     elif d < max(neighbors.keys()):
        #         neighbors.pop(max(neighbors.keys()))
        #         neighbors.update({d:train_ring})
        # return int(round(sum(neighbors.values())/len(neighbors)))

In [109]:
import pandas as pd
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo # access to our datasets
from sklearn.preprocessing import OneHotEncoder # access to preprocessing tool

# fetch dataset
abalone = fetch_ucirepo(id=1)

# data (as pandas dataframes)
# iloc "slicing" reduces the number of features + entries
X = abalone.data.features.iloc[0:1000, 0:3]
y = abalone.data.targets.iloc[0:1000]

# Apply One-Hot Encoding to the 'Sex' column

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Encode only the 'Sex' column
sex_encoded = encoder.fit_transform(X[['Sex']])

# Create a new DataFrame for the encoded 'Sex' column with appropriate column names
# get_feauture_names_out identifies all unique categories in the 'Sex' column.
sex_df = pd.DataFrame(sex_encoded,
                      columns=encoder.get_feature_names_out(['Sex']),
                      index=X.index)

# Drop the original 'Sex' column from X and concatenate the encoded 'Sex' DataFrame
X = pd.concat([X.drop('Sex', axis=1), sex_df], axis=1)

In [110]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# How many feautres (cols) and entries (rows) are in the training data?
print("Training features shape:", X_train.shape)

# How many labels/answers do we have to train our model on?
print("Training labels shape:", y_train.shape)

# Same info as above, for texting the accuracy of our model.
print("Testing features shape:", X_test.shape)
print("Testing labels shape:", y_test.shape)

Training features shape: (800, 5)
Training labels shape: (800, 1)
Testing features shape: (200, 5)
Testing labels shape: (200, 1)


In [ ]:
import pandas as pd # Keep import if other pandas operations are intended here
# Removed OneHotEncoder import and usage as encoding is now done earlier

trials = []
accuracies = []
tasks = ['classification', 'regression', 'ordered regression', 'weighted regression']

knn = KNN()
# Fit the custom KNN model with the already encoded numerical data
# X_train and X_test are now numerical because X was encoded earlier before the split.
# y_train.values are the 'Rings' which are numerical and used as labels for classification.
knn.fit(X_train.values, y_train.values)

# test a range of k values for each task
for k in range(1,500):
    knn.set_k(k)
    for i in range(len(tasks)):
        knn.set_task(tasks[i])

        # Make predictions using the already encoded test data
        y_pred = knn.predict(X_test.values)

In [111]:
# # Calculate accuracy

# from sklearn.metrics import accuracy_score

# accuracy_custom = accuracy_score(y_test, y_pred)
# print("Custom Accuracy:", accuracy_custom)

# Find most accurate tasks and k-values
accurate_indices = list(np.argsort(accuracies))
accurate_indices.reverse()
for i in accurate_indices[:100]:
    print(trials[i])
    print(accuracies[i])

regression, k=60
0.215
regression, k=59
0.215
regression, k=75
0.21
regression, k=79
0.21
regression, k=58
0.21
regression, k=69
0.205
regression, k=80
0.205
regression, k=78
0.205
regression, k=65
0.205
regression, k=62
0.205
regression, k=63
0.205
regression, k=68
0.205
ordered regression, k=86
0.2
ordered regression, k=83
0.2
ordered regression, k=92
0.2
ordered regression, k=91
0.2
ordered regression, k=84
0.2
ordered regression, k=85
0.2
ordered regression, k=87
0.2
ordered regression, k=77
0.2
regression, k=76
0.2
regression, k=77
0.2
regression, k=74
0.2
regression, k=73
0.2
regression, k=71
0.2
ordered regression, k=76
0.2
regression, k=53
0.2
regression, k=61
0.2
regression, k=57
0.2
regression, k=70
0.2
regression, k=67
0.2
regression, k=66
0.2
regression, k=64
0.2
regression, k=56
0.2
ordered regression, k=82
0.2
ordered regression, k=98
0.2
ordered regression, k=81
0.2
ordered regression, k=80
0.2
ordered regression, k=96
0.2
ordered regression, k=93
0.2
ordered regression,

In [ ]:
# Create and train the KNN model, to compare our accuracy to that of SKLearn's model
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=9)
knn.fit(X_train, y_train)

# Make predictions on the test set
y_pred = knn.predict(X_test)

# Calculate accuracy
accuracy_sklearn = accuracy_score(y_test, y_pred)
print("Sklearn Accuracy:", accuracy_sklearn)

Sklearn Accuracy: 0.14


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
